# 🪞 The Reflection Pattern from Scratch

## Learning Objectives
In this notebook, you will learn:
1. **The Generation Step** - Ask the model for a first draft with a plain chat-completions call
2. **The Reflection Step** - Give the model a second persona whose only job is to critique that draft
3. **Closing the Loop** - Feed the critique back as a user message so the next draft improves
4. **A Reflection Agent** - Wrap the generate → reflect cycle in a class with a bounded history and an early stop

## Where this fits
This is the first notebook of the course's from-scratch track: no framework, just the OpenAI SDK and Python. `02_tool_use_pattern` adds tools, `03_planning_pattern_react` turns tools into a loop, and `04_real_world_agent_hacker_news_chatbot` packages all three into a small app.

## Prerequisites
- Python and a basic chat-completions API call
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📖 What is the Reflection Pattern?

The first pattern we are going to implement is the **reflection pattern**.


<img src="../assets/img/reflection_pattern.png" alt="Reflection pattern diagram" width="600"/>


This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` from the project-root `.env`; the next cell creates the raw OpenAI client every step below talks to.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM CLIENT: the raw OpenAI SDK, no framework
# ============================================================================
# This module builds agents from scratch, so it talks to the chat-completions API
# directly instead of going through LangChain.
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"

print(f"✅ OpenAI client ready, model: {MODEL}")

✅ OpenAI client ready, model: gpt-4o-mini


---
## ✍️ Part 1: The Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

In this example, we'll test the Python coding skills of `gpt-4o-mini` (the model every notebook in this module uses). In particular, we are going to ask the LLM to code a famous sorting algorithm: **Merge Sort**.

---

<img src="../assets/img/mergesort.png" alt="Merge sort" width="400"/>

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python
programmer eager to receive feedback / critique by the user.

In [3]:
# ============================================================================
# GENERATION HISTORY: system prompt + the user's request
# ============================================================================
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique,"
        "respond with a revised version of your previous attempt."
    }
]

generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

for message in generation_chat_history:
    print(f"[{message['role']}] {message['content'][:100]}")

[system] You are a Python programmer tasked with generating high quality Python code.Your task is to Generate
[user] Generate a Python implementation of the Merge Sort algorithm


Let's generate the first version of the merge sort implementation.

In [4]:
# ============================================================================
# GENERATE: first draft
# ============================================================================
from IPython.display import display_markdown

mergesort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model=MODEL
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

display_markdown(mergesort_code, raw=True)

Here's a clean and efficient implementation of the Merge Sort algorithm in Python. Merge Sort is a divide-and-conquer algorithm that splits the list into halves, recursively sorts them, and then merges the sorted halves back together.

Here's the code:

```python
def merge_sort(arr):
    # Base case: if the array is of length 1 or empty, it's already sorted
    if len(arr) <= 1:
        return arr

    # Finding the midpoint of the array
    mid = len(arr) // 2

    # Recursively sorting the left and right halves
    left_half = merge_sort(arr[:mid])
    right_half = merge_sort(arr[mid:])

    # Merging the sorted halves
    return merge(left_half, right_half)

def merge(left, right):
    sorted_array = []
    left_index, right_index = 0, 0

    # Merge the two halves while there are elements in both
    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            sorted_array.append(left[left_index])
            left_index += 1
        else:
            sorted_array.append(right[right_index])
            right_index += 1

    # If there are any remaining elements in left_half or right_half, add them
    sorted_array.extend(left[left_index:])
    sorted_array.extend(right[right_index:])

    return sorted_array

# Example usage:
if __name__ == "__main__":
    array = [38, 27, 43, 3, 9, 82, 10]
    sorted_array = merge_sort(array)
    print("Sorted array:", sorted_array)
```

### Explanation:
1. **`merge_sort(arr)` function**: This function checks if the input array is either empty or has one element. If so, it returns the array directly (base case). Otherwise, it splits the array into two halves, recursively sorts each half, and merges the sorted halves.

2. **`merge(left, right)` function**: This helper function merges two sorted arrays into a single sorted array. It iterates through both arrays, comparing their elements, and appending the smaller element to the result array. After one of the arrays is exhausted, any remaining elements from the other array are added to the result.

3. **Example usage**: The given code includes an example that sorts a predefined array and prints the sorted result.

You can run this code block in any Python environment to see how it works!

---
## 🧐 Part 2: The Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)

<img src="../assets/img/karpathy.png" alt="Andrej Karpathy" width="400"/>

The user message, in this case,  is the code generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [5]:
# ============================================================================
# REFLECTION HISTORY: critic persona + the draft to critique
# ============================================================================
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

print(f"✅ Reflection history has {len(reflection_chat_history)} messages")

✅ Reflection history has 2 messages


Now, let's generate a critique to the Python code.

In [6]:
# ============================================================================
# REFLECT: critique the draft
# ============================================================================
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model=MODEL
).choices[0].message.content

display_markdown(critique, raw=True)

Your implementation of the Merge Sort algorithm in Python is indeed clean and efficient. The use of recursion and the clear separation of concerns between the sorting and merging processes is commendable. Here’s a critique and some recommendations for improvement:

### Critique:

1. **Performance**: The algorithm has a time complexity of \(O(n \log n)\), which is optimal for a comparison-based sorting algorithm. However, be mindful that the space complexity is \(O(n)\) due to the use of additional arrays.

2. **Code Clarity**: While the code is quite straightforward, some comments are slightly redundant. Comments should ideally add information that is not immediately obvious from the code itself.

3. **Type Annotations**: Python supports type hints which can improve code readability and help with static type-checking. Adding type annotations to the functions would make the code clearer about expected input and output types.

4. **In-place Sorting**: The current implementation uses additional space for merging the arrays. While not necessary, it's worth noting that there are in-place sorting algorithms that could be explored if space efficiency becomes critical.

5. **Testing**: The example usage helps demonstrate the function, but consider adding some assertions or test cases to validate the correctness of the implementation systematically.

### Recommendations:

1. **Implement Type Annotations**:
   ```python
   def merge_sort(arr: list[int]) -> list[int]:
   ```

2. **Reduce Redundant Comments**: Consider revising or removing comments that repeat what the code does. For example, "Finding the midpoint of the array" could be implied just from the code itself.

3. **Testing**: Implement a few more test cases to ensure robustness. For example:
   ```python
   assert merge_sort([]) == []
   assert merge_sort([1]) == [1]
   assert merge_sort([3, 2]) == [2, 3]
   assert merge_sort([5, 4, 3, 2, 1]) == [1, 2, 3, 4, 5]
   ```

4. **Consider Edge Cases**: Think about how the function will perform with large data sets or already sorted lists, and perhaps add some timing code to measure performance.

5. **Documentation**: Including a docstring at the beginning of your function (using the triple quotes style) summarizing its purpose, parameters, and return value would improve usability and maintainability.

Here's the updated `merge_sort` function signature with type annotations and a basic docstring added:

```python
def merge_sort(arr: list[int]) -> list[int]:
    """
    Perform merge sort on a list of integers.

    Parameters:
    arr (list[int]): A list of integers to be sorted.

    Returns:
    list[int]: A sorted list of integers.
    """
    if len(arr) <= 1:
        return arr
    ...
```

In conclusion, your implementation of Merge Sort is solid! Emphasizing clarity and robustness through testing and the use of Python's typing features will serve you well as your projects grow in complexity. Keep up the great work!

---
## 🔁 Part 3: Closing the Loop

Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role. Then we generate again: the second draft answers the critique.

In [7]:
# ============================================================================
# REGENERATE: second draft that answers the critique
# ============================================================================
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

essay = client.chat.completions.create(
    messages=generation_chat_history,
    model=MODEL
).choices[0].message.content

display_markdown(essay, raw=True)

Thank you for the valuable feedback! Based on your critique and recommendations, I have revised the Merge Sort implementation to enhance clarity, performance, and robustness. Here is the updated version:

```python
from typing import List

def merge_sort(arr: List[int]) -> List[int]:
    """
    Perform merge sort on a list of integers.

    Parameters:
    arr (List[int]): A list of integers to be sorted.

    Returns:
    List[int]: A sorted list of integers.
    """
    # Base case: if the array is of length 1 or empty, it's already sorted
    if len(arr) <= 1:
        return arr

    # Finding the midpoint of the array
    mid = len(arr) // 2

    # Recursively sorting the left and right halves
    left_half = merge_sort(arr[:mid])
    right_half = merge_sort(arr[mid:])

    # Merging the sorted halves
    return merge(left_half, right_half)

def merge(left: List[int], right: List[int]) -> List[int]:
    """
    Merge two sorted lists into a single sorted list.

    Parameters:
    left (List[int]): The first sorted list.
    right (List[int]): The second sorted list.

    Returns:
    List[int]: A merged sorted list containing all elements from both lists.
    """
    sorted_array = []
    left_index, right_index = 0, 0

    # Merge the two halves while there are elements in both
    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            sorted_array.append(left[left_index])
            left_index += 1
        else:
            sorted_array.append(right[right_index])
            right_index += 1

    # If there are any remaining elements in left_half or right_half, add them
    sorted_array.extend(left[left_index:])
    sorted_array.extend(right[right_index:])

    return sorted_array

# Example usage with tests
if __name__ == "__main__":
    test_cases = [
        ([38, 27, 43, 3, 9, 82, 10], [3, 9, 10, 27, 38, 43, 82]),
        ([], []),
        ([1], [1]),
        ([3, 2], [2, 3]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
        ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5])  # Already sorted
    ]

    for arr, expected in test_cases:
        assert merge_sort(arr) == expected

    print("All test cases passed!")
```

### Changes Made:
1. **Type Annotations**: Added type hints to both the `merge_sort` and `merge` functions to clarify the expected input and output types.

2. **Docstrings**: Included docstrings for both functions to explain their purpose, parameters, and return values, improving code documentation.

3. **Testing**: Added a set of test cases at the bottom of the script to validate the correctness of the implementation. This will help catch any potential issues and reassure users that the function works as intended.

4. **Reduced Comments**: Kept comments to a minimum while ensuring they still add value to understanding the code.

This revised implementation maintains the integrity of the Merge Sort algorithm while addressing the points you raised, making it clearer and more robust. Feel free to provide further feedback or additional requests!

### And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. The agent below does both: it iterates for at most `n_steps` rounds and stops early when the critic answers `<OK>`.

---
## 🤖 Part 4: Building a Reflection Agent

Now that you understand the loop, let's implement it as a class. First, a few helpers: prompt builders, a `ChatHistory` that is a plain Python list with an optional maximum length, and a `FixedFirstChatHistory` that always keeps its first message (the system prompt) when it drops old ones.

In [8]:
# ============================================================================
# HELPERS: prompt builders, bounded chat histories, pretty printing
# ============================================================================
import time

from colorama import Fore, Style


def completions_create(client, messages: list, model: str) -> str:
    """
    Sends a request to the client's `completions.create` method to interact with the language model.

    Args:
        client (OpenAI): The OpenAI client object
        messages (list[dict]): A list of message objects containing chat history for the model.
        model (str): The model to use for generating tool calls and responses.

    Returns:
        str: The content of the model's response.
    """
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt (str): The actual content of the prompt.
        role (str): The role of the speaker (e.g., user, assistant).

    Returns:
        dict: A dictionary representing the structured prompt.
    """
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history: list, msg: str, role: str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history (list): The list representing the current chat history.
        msg (str): The message to append.
        role (str): The role type (e.g. 'user', 'assistant', 'system')
    """
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg: str):
        """Add a message to the queue.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)



class FixedFirstChatHistory(ChatHistory):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        super().__init__(messages, total_length)

    def append(self, msg: str):
        """Add a message to the queue. The first messaage will always stay fixed.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(1)
        super().append(msg)

def fancy_print(message: str) -> None:
    """
    Displays a fancy print message.

    Args:
        message (str): The message to display.
    """
    print(Style.BRIGHT + Fore.CYAN + f"\n{'=' * 50}")
    print(Fore.MAGENTA + f"{message}")
    print(Style.BRIGHT + Fore.CYAN + f"{'=' * 50}\n")
    time.sleep(0.5)


def fancy_step_tracker(step: int, total_steps: int) -> None:
    """
    Displays a fancy step tracker for each iteration of the generation-reflection loop.

    Args:
        step (int): The current step in the loop.
        total_steps (int): The total number of steps in the loop.
    """
    fancy_print(f"STEP {step + 1}/{total_steps}")

print("✅ Helpers defined")

✅ Helpers defined


The agent alternates `generate()` and `reflect()` for `n_steps` rounds. Both histories are capped at three messages (system prompt + the last exchange), so long loops don't blow up the context. The reflection prompt asks the critic to answer `<OK>` when nothing is left to fix, which ends the loop early.

In [9]:
# ============================================================================
# REFLECTION AGENT: generate → reflect for n_steps rounds
# ============================================================================
BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = """
You are tasked with generating critique and recommendations to the user's generated content.
If the user content has something wrong or something to be improved, output a list of recommendations
and critiques. If the user content is ok and there's nothing to change, output this: <OK>
"""


class ReflectionAgent:
    """
    A class that implements a Reflection Agent, which generates responses and reflects
    on them using the LLM to iteratively improve the interaction. The agent first generates
    responses based on provided prompts and then critiques them in a reflection step.

    Attributes:
        model (str): The model name used for generating and reflecting on responses.
        client (OpenAI): An instance of the OpenAI client to interact with the language model.
    """

    def __init__(self, model: str = MODEL):
        self.client = OpenAI()  # reads OPENAI_API_KEY from the environment
        self.model = model

    def _request_completion(
        self,
        history: list,
        verbose: int = 0,
        log_title: str = "COMPLETION",
        log_color: str = "",
    ):
        """
        A private method to request a completion from the OpenAI model.

        Args:
            history (list): A list of messages forming the conversation or reflection history.
            verbose (int, optional): The verbosity level. Defaults to 0 (no output).

        Returns:
            str: The model-generated response.
        """
        output = completions_create(self.client, history, self.model)

        if verbose > 0:
            print(log_color, f"\n\n{log_title}\n\n", output)

        return output

    def generate(self, generation_history: list, verbose: int = 0) -> str:
        """
        Generates a response based on the provided generation history using the model.

        Args:
            generation_history (list): A list of messages forming the conversation or generation history.
            verbose (int, optional): The verbosity level, controlling printed output. Defaults to 0.

        Returns:
            str: The generated response.
        """
        return self._request_completion(
            generation_history, verbose, log_title="GENERATION", log_color=Fore.BLUE
        )

    def reflect(self, reflection_history: list, verbose: int = 0) -> str:
        """
        Reflects on the generation history by generating a critique or feedback.

        Args:
            reflection_history (list): A list of messages forming the reflection history, typically based on
                                       the previous generation or interaction.
            verbose (int, optional): The verbosity level, controlling printed output. Defaults to 0.

        Returns:
            str: The critique or reflection response from the model.
        """
        return self._request_completion(
            reflection_history, verbose, log_title="REFLECTION", log_color=Fore.GREEN
        )

    def run(
        self,
        user_msg: str,
        generation_system_prompt: str = "",
        reflection_system_prompt: str = "",
        n_steps: int = 10,
        verbose: int = 0,
    ) -> str:
        """
        Runs the ReflectionAgent over multiple steps, alternating between generating a response
        and reflecting on it for the specified number of steps.

        Args:
            user_msg (str): The user message or query that initiates the interaction.
            generation_system_prompt (str, optional): The system prompt for guiding the generation process.
            reflection_system_prompt (str, optional): The system prompt for guiding the reflection process.
            n_steps (int, optional): The maximum number of generate-reflect cycles. Defaults to 10.
            verbose (int, optional): The verbosity level controlling printed output. Defaults to 0.

        Returns:
            str: The final generated response after all cycles are completed.
        """
        generation_system_prompt += BASE_GENERATION_SYSTEM_PROMPT
        reflection_system_prompt += BASE_REFLECTION_SYSTEM_PROMPT

        # Given the iterative nature of the Reflection Pattern, we might exhaust the LLM context (or
        # make it really slow). That's the reason I'm limitting the chat history to three messages.
        # The `FixedFirstChatHistory` is a very simple class, that creates a Queue that always keeps
        # fixed the first message. I thought this would be useful for maintaining the system prompt
        # in the chat history.
        generation_history = FixedFirstChatHistory(
            [
                build_prompt_structure(prompt=generation_system_prompt, role="system"),
                build_prompt_structure(prompt=user_msg, role="user"),
            ],
            total_length=3,
        )

        reflection_history = FixedFirstChatHistory(
            [build_prompt_structure(prompt=reflection_system_prompt, role="system")],
            total_length=3,
        )

        for step in range(n_steps):
            if verbose > 0:
                fancy_step_tracker(step, n_steps)

            # Generate the response
            generation = self.generate(generation_history, verbose=verbose)
            update_chat_history(generation_history, generation, "assistant")
            update_chat_history(reflection_history, generation, "user")

            # Reflect and critique the generation
            critique = self.reflect(reflection_history, verbose=verbose)

            # Early stop: the critic answers <OK> when it has nothing left to improve
            if "<OK>" in critique:
                print(Fore.RED, "\n\nStop sequence found. Stopping the reflection loop ...\n\n")
                break

            update_chat_history(generation_history, critique, "user")
            update_chat_history(reflection_history, critique, "assistant")

        return generation

print("✅ ReflectionAgent defined")

✅ ReflectionAgent defined


A lenient critic answers `<OK>` right away and the loop ends after one round, so the reflection persona below is told to be demanding. Try the plain `"You are Andrej Karpathy, an experienced computer scientist"` persona afterwards and compare how many rounds run.

In [10]:
# ============================================================================
# RUN: up to three rounds of generate → reflect
# ============================================================================
agent = ReflectionAgent()

generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = (
    "You are Andrej Karpathy, an experienced computer scientist and a demanding code reviewer. "
    "Always look for concrete improvements (docstrings, type hints, edge cases, tests, performance) "
    "before approving code. "
)

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3,
    verbose=1,
)


STEP 1/3



 

GENERATION

 Here's a Python implementation of the Merge Sort algorithm:

```python
def merge_sort(array):
    # If the array is of length 1 or empty, it is already sorted
    if len(array) <= 1:
        return array
    
    # Find the middle point to divide the array into two halves
    mid = len(array) // 2
    
    # Recursively sort both halves
    left_half = merge_sort(array[:mid])
    right_half = merge_sort(array[mid:])
    
    # Merge the sorted halves
    return merge(left_half, right_half)

def merge(left, right):
    sorted_array = []
    left_index, right_index = 0, 0
    
    # Compare elements from left and right arrays and merge them
    while left_index < len(left) and right_index < len(right):
        if left[left_index] < right[right_index]:
            sorted_array.append(left[left_index])
            left_index += 1
        else:
            sorted_array.append(right[right_index])
            right_index += 1
    
    # If there are remaining elements in the l

 

REFLECTION

 Here are my recommendations and critiques for your implementation of the Merge Sort algorithm:

1. **Docstrings**: 
   - Include docstrings for both `merge_sort` and `merge` functions that describe the input parameters, return types, and the overall functionality. This greatly enhances readability and maintainability.

   ```python
   def merge_sort(array: list[int]) -> list[int]:
       """
       Sorts an input array in ascending order using the merge sort algorithm.

       Parameters:
       array (list[int]): The array of integers to be sorted.

       Returns:
       list[int]: A new sorted array in ascending order.
       """
   ```

   ```python
   def merge(left: list[int], right: list[int]) -> list[int]:
       """
       Merges two sorted arrays into a single sorted array.

       Parameters:
       left (list[int]): The first sorted array.
       right (list[int]): The second sorted array.

       Returns:
       list[int]: A merged and sorted array containi

 

GENERATION

 Thank you for your thorough critique and valuable suggestions. Here's the revised implementation of the Merge Sort algorithm addressing your recommendations:

```python
def merge_sort_algorithm(array: list[int]) -> list[int]:
    """
    Sorts an input array in ascending order using the merge sort algorithm.

    Parameters:
    array (list[int]): The array of integers to be sorted.

    Returns:
    list[int]: A new sorted array in ascending order.
    """
    # Check if the input is a list
    if not isinstance(array, list):
        raise TypeError("Input must be a list.")
    
    # If the array is of length 1 or empty, it is already sorted
    if len(array) <= 1:
        return array
    
    # Find the middle point to divide the array into two halves
    mid = len(array) // 2
    
    # Recursively sort both halves
    left_half = merge_sort_algorithm(array[:mid])
    right_half = merge_sort_algorithm(array[mid:])
    
    # Merge the sorted halves
    return merge

 

REFLECTION

 <OK>
 

Stop sequence found. Stopping the reflection loop ...




In [11]:
# ============================================================================
# RESULT: the final draft
# ============================================================================
display_markdown(final_response, raw=True)

Thank you for your thorough critique and valuable suggestions. Here's the revised implementation of the Merge Sort algorithm addressing your recommendations:

```python
def merge_sort_algorithm(array: list[int]) -> list[int]:
    """
    Sorts an input array in ascending order using the merge sort algorithm.

    Parameters:
    array (list[int]): The array of integers to be sorted.

    Returns:
    list[int]: A new sorted array in ascending order.
    """
    # Check if the input is a list
    if not isinstance(array, list):
        raise TypeError("Input must be a list.")
    
    # If the array is of length 1 or empty, it is already sorted
    if len(array) <= 1:
        return array
    
    # Find the middle point to divide the array into two halves
    mid = len(array) // 2
    
    # Recursively sort both halves
    left_half = merge_sort_algorithm(array[:mid])
    right_half = merge_sort_algorithm(array[mid:])
    
    # Merge the sorted halves
    return merge(left_half, right_half)

def merge(left: list[int], right: list[int]) -> list[int]:
    """
    Merges two sorted arrays into a single sorted array.

    Parameters:
    left (list[int]): The first sorted array.
    right (list[int]): The second sorted array.

    Returns:
    list[int]: A merged and sorted array containing elements from both left and right.
    """
    sorted_array = []
    left_index, right_index = 0, 0
    
    # Compare elements from left and right arrays and merge them
    while left_index < len(left) and right_index < len(right):
        if left[left_index] < right[right_index]:
            sorted_array.append(left[left_index])
            left_index += 1
        else:
            sorted_array.append(right[right_index])
            right_index += 1
    
    # If there are remaining elements in the left array, add them
    while left_index < len(left):
        sorted_array.append(left[left_index])
        left_index += 1
    
    # If there are remaining elements in the right array, add them
    while right_index < len(right):
        sorted_array.append(right[right_index])
        right_index += 1
        
    return sorted_array

# Example usage with edge cases
def run_example():
    test_cases = [
        [38, 27, 43, 3, 9, 82, 10],  # Normal case
        [],                          # Edge case: empty list
        [1],                         # Edge case: single element
        [5, 4, 3, 2, 1],            # Edge case: reverse sorted
        [1, 2, 3, 3, 2, 1],         # Edge case: duplicates
        [4, 2, 6, 1, 3, 5],         # Normal case
    ]
    
    for i, case in enumerate(test_cases, 1):
        try:
            sorted_result = merge_sort_algorithm(case)
            print(f"Test case {i}: {case} -> Sorted: {sorted_result}")
        except Exception as e:
            print(f"Test case {i}: {case} resulted in an error: {e}")

if __name__ == "__main__":
    run_example()
```

### Changes Made:
1. **Docstrings**: Added comprehensive docstrings for both functions.
2. **Type Hints**: Applied specific type hints for function parameters and return types.
3. **Error Handling**: Included a type check to raise a `TypeError` if the input is not a list.
4. **Example Usage**: Reorganized the example into a function with various test cases to demonstrate robustness.
5. **Function Naming**: Changed the name of the `merge_sort` function to `merge_sort_algorithm` for greater clarity.

This implementation should now provide better readability, maintainability, and usability. Let me know if there's anything else you'd like to improve!

With `verbose=1` the run above printed every GENERATION and REFLECTION: each new generation answers the critique before it. If a REFLECTION printed `<OK>`, the loop stopped there because the critic had nothing left to fix; otherwise it ran all `n_steps` rounds and returned the last draft.

---
## 📝 Summary

In this notebook, we learned:

### 1. The Generation Step
- A generation is one chat-completions call over a history that starts with a system prompt and the user's request

### 2. The Reflection Step
- A second history with a critic persona turns the draft into the *user* message, so the model critiques rather than continues it

### 3. Closing the Loop
- The critique goes back into the generation history as a user message; the next draft answers it

### 4. A Reflection Agent
- `ReflectionAgent.run()` repeats generate → reflect for up to `n_steps` rounds
- Bounded histories (`FixedFirstChatHistory`) keep the context small; an `<OK>` stop sequence ends the loop when there is nothing left to improve

### Next Steps
- **`02_tool_use_pattern`**: give the model access to the outside world by describing Python functions in the prompt and running the calls it asks for